In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual para las gráficas
sns.set_theme(style="whitegrid")

# 1. Cargar los datos
# Asegúrate de que el archivo 'hotel_bookings.csv' esté en la misma carpeta
df = pd.read_csv('hotel-cancellation-prediction/data/raw/hotel_bookings.csv')

# 2. Vista previa rápida
print("--- TAMAÑO DEL DATASET ---")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print("\n--- PRIMERAS 5 FILAS ---")
display(df.head())

# 3. Diagnóstico de Tipos de Datos y Nulos
print("\n--- INFORMACIÓN DE COLUMNAS Y NULOS ---")
df.info()

# 4. Conteo exacto de valores nulos
print("\n--- VALORES FALTANTES POR COLUMNA ---")
nulos = df.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

#5. Estadísticas descriptivas
df.describe(include='all').T

#6. Pruebas de valores anormales
huespedes_fantasma = (df['adults'] > 0) | (df['children'] > 0) | (df['babies'] > 0)
df_prueba = df[huespedes_fantasma].copy()
print(f"Filas eliminadas (reservas fantasma): {df.shape[0] - df_prueba.shape[0]}")
print(f"Total de filas finales: {df_prueba.shape[0]}")


Observamos que tenemos mas de 100,000 filas y 32 columnas, de las cuales tambien detectamos anomalias en el tipo de datos y valores nulos, ademas de encontrar valores fantasma (reservas sin ninguna persona).
Primero manejemos los valores nulos:
- Company tiene mas de 80% de datos faltantes, en esta variable lo normal seria el borrado de la columna, pero en nuestro caso utilizaremos la ausencia de esos datos como una caracteristica predictiva, conviertiendo la variable en binaria.
- Agent(16,340 nulos) Esta columna representa el ID de la agencia de vaijes, por lo tanto no hay errores, si no que el cliente hizo la reserva directamente con el hotel. Rellenaremos los nulos con 0.
- Country (488 nulos) Esta es una variable categrica la cual no podemos hacer una imputacion estadistica, lo mejor en este caso es rellenar con 'Unknown'.
- Children (4 nulos) Rellenaremos con la mediana natural (0)

En cuanto a la coreccion de Tipos convertiremos children y agent de tipos float a enteros ya que estos valores no pueden tener decimales.
Tambien las 180 reservas fantasma encontradas se eliminaran.


In [ ]:
# Verificacion de los datos Limpios(El dataset proviene de la limpieza realizada en el notebook 01_hotel_eda.ipynb)
df_clean = pd.read_csv('hotel-cancellation-prediction/data/processed/hotel_bookings_clean_v1.csv')
print("--- VALORES FALTANTES DESPUÉS DE LA LIMPIEZA ---")
print(df_clean.isnull().sum().max()) 


## Investigacion de Variables
Antes de avanzar con el modelo necesitamos terminar el procesamiento de datos con la seleccion de variables, para esto tenemos en cuenta dos conceptos "Data Leakage" y la Capacidad de Generalizacion.

Para un modelo predictivo no podemos tener informacion del futuro o pasado, ya que el modelo haria trampa. Solo podemos darle informacion que el hotel tendra disponible en el momento que el cliente haga la reserva ya que es el momento justo donde el modelo actua, entonces descartamos las siguientes variables:
- reservation_status: Esta columna tiene valores como "Canceled", "Check-Out" o "No-Show". Si le pasamos esto al algoritmo, el modelo hará trampa.
- reservation_status_date: Esta columna tiene como valor la fecha en la que el cliente cancelo o hizo check-out.
- arrival_date_year: Descartando esta variable evitamos que el modelo haga reglas como "Si el año es 2015, hay mas cancelaciones" ya que es algo inutil.
- A diferencia de la variable año, dejaremos las variables de mes y semana ya que pueden generar una prediccion de comportamiento, debido a las temporadas altas y bajas.

### Multicolinealidad
Buscamos las variables que podrian tener Multicolinealidad

In [ ]:

columnas_a_eliminar = ['reservation_status', 'reservation_status_date', 'arrival_date_year']
df_model = df_clean.drop(columnas_a_eliminar, axis=1)
X = df_model.drop('is_canceled', axis=1)
y = df_model['is_canceled']
# 1. Aislar solo las variables numéricas
X_num = X.select_dtypes(include=['int64', 'float64'])

# 2. Calcular la matriz de correlación (Método Pearson)
corr_matrix = X_num.corr()

# 3. Dibujar el Mapa de Calor (Heatmap)
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', 
            vmin=-1, vmax=1, square=True, linewidths=.5)
plt.title('Matriz de Correlación de Variables Numéricas', fontsize=16, fontweight='bold')
plt.show()
# Para una lectura facil hacemos una lista con aquellas variables que tienen una correlación mayor a 0.80
umbral = 0.80 
print(f"--- PARES DE VARIABLES CON CORRELACIÓN MAYOR A {umbral} ---")
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > umbral:
            colname_1 = corr_matrix.columns[i]
            colname_2 = corr_matrix.columns[j]
            correlacion = corr_matrix.iloc[i, j]
            print(f"⚠️ {colname_1} y {colname_2} (Correlación: {correlacion:.2f})")

Al leer la grafica llegamos a la conclusion de que tenemos Señales Independientes, la correlacion mas alta es de apenas 0.49, no tenemos multicolinealidad grave y podemos conservar todas las variables ya que cada una de ellas aporta informacion unica al modelo.

### Cardinalidad
Verificamos las variables categoricos, en especifico la cardinalidad de estas, ya que una variable con una alta cantidad de categorias (alta cardinalidad) consume memoria innesesaria.

In [ ]:
# Seleccionamos solo las variables de texto (object)
X_cat = X.select_dtypes(include=['object'])

print("--- CANTIDAD DE CATEGORÍAS ÚNICAS POR COLUMNA ---")
for col in X_cat.columns:
    unicos = X_cat[col].nunique()
    if unicos > 15:
        print(f"PELIGRO - {col}: {unicos} categorías (Alta Cardinalidad)")
    else:
        print(f" OK - {col}: {unicos} categorías")

Podemos notar que solo la variable de paises tiene una alta cardinalidad, limpiaremos esta variable gracias a su naturaleza. Habra un grupo de paises con grandes cantidades de reservas, de ahi en mas un gran porcentaje solo tendra una o un par, lo que en genral no afecta ni aporta mucho. Haremos una reduccion a 10 + 1 categorias, los 10 paises con mas trafico y todos los demas en una categoria llamada Other. 